# AnalyticsPath Python Analytics Lab - Solution


In [1]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import plotly.express as px

pd.set_option("display.float_format", lambda x: f"{x:,.2f}")
DATA = Path("../data/powerbi/retail_sales_powerbi.csv")
df = pd.read_csv(DATA, parse_dates=["Order_Date"])
df.head()


,Order_ID,Order_Date,Region,City,Category,Subcategory,Product,Customer_Segment,Channel,Salesperson,...,Unit_Price,Unit_Cost,Discount,Revenue,Cost,Profit,Profit_Margin,Year,Month,Year_Month
0,ORD-00001,2025-10-16,North,Kaduna,Fashion,Apparel,Classic Shirt,Consumer,Online,Amina Yusuf,...,39.92,18.43,0.05,113.77,55.29,58.48,0.51,2025,October,2025-10
1,ORD-00002,2024-04-05,South West,Lagos,Electronics,Phones,ProMax 12,Corporate,Retail Store,Chinedu Okafor,...,661.68,518.74,0.15,"3,937.00","3,631.18",305.82,0.08,2024,April,2024-04
2,ORD-00003,2025-12-15,South South,Warri,Fashion,Apparel,Classic Shirt,Small Business,Retail Store,Amina Yusuf,...,36.30,18.01,0.00,217.80,108.06,109.74,0.50,2025,December,2025-12
3,ORD-00004,2025-04-15,North,Kano,Electronics,Accessories,Smart Watch,Small Business,Retail Store,Amina Yusuf,...,133.46,86.21,0.00,800.76,517.26,283.50,0.35,2025,April,2025-04
4,ORD-00005,2024-08-26,North,Kano,Fashion,Footwear,Casual Sandal,Consumer,Distributor,Ngozi Obi,...,40.50,20.97,0.10,218.70,125.82,92.88,0.42,2024,August,2024-08


## 1. Data quality


In [ ]:
quality = {
    "rows": len(df),
    "columns": df.shape[1],
    "missing_cells": int(df.isna().sum().sum()),
    "duplicate_order_ids": int(df["Order_ID"].duplicated().sum()),
    "min_date": df["Order_Date"].min(),
    "max_date": df["Order_Date"].max(),
}
quality


## 2. Executive KPIs


In [ ]:
kpis = {
    "Total Revenue": df["Revenue"].sum(),
    "Total Profit": df["Profit"].sum(),
    "Profit Margin": df["Profit"].sum()/df["Revenue"].sum(),
    "Orders": df["Order_ID"].nunique(),
    "Units Sold": df["Quantity"].sum(),
    "Average Order Value": df["Revenue"].mean(),
}
pd.Series(kpis)


## 3. Regional and category performance


In [ ]:
region_summary = df.groupby("Region", as_index=False).agg(
    Revenue=("Revenue","sum"),
    Profit=("Profit","sum"),
    Orders=("Order_ID","nunique")
)
region_summary["Profit_Margin"] = region_summary["Profit"]/region_summary["Revenue"]
region_summary.sort_values("Revenue", ascending=False)


In [ ]:
category_summary = df.groupby("Category", as_index=False).agg(
    Revenue=("Revenue","sum"), Profit=("Profit","sum")
).sort_values("Revenue", ascending=False)
category_summary


## 4. Monthly trend


In [ ]:
monthly = df.groupby("Year_Month", as_index=False).agg(
    Revenue=("Revenue","sum"), Profit=("Profit","sum")
).sort_values("Year_Month")
px.line(monthly, x="Year_Month", y="Revenue", markers=True, title="Monthly Revenue").show()


## 5. Segment and channel


In [ ]:
segment = df.groupby("Customer_Segment", as_index=False).agg(
    Revenue=("Revenue","sum"),
    Profit=("Profit","sum"),
    Orders=("Order_ID","nunique"),
    Avg_Order_Value=("Revenue","mean")
).sort_values("Revenue", ascending=False)
segment


In [ ]:
channel = df.groupby("Channel", as_index=False).agg(
    Revenue=("Revenue","sum"),
    Profit=("Profit","sum"),
    Orders=("Order_ID","nunique"),
    Avg_Order_Value=("Revenue","mean")
).sort_values("Revenue", ascending=False)
channel


## 6. Discount and profitability


In [ ]:
discount_summary = df.groupby("Discount", as_index=False).agg(
    Orders=("Order_ID","nunique"),
    Avg_Profit_Margin=("Profit_Margin","mean"),
    Profit=("Profit","sum")
)
discount_summary["Discount_Pct"] = discount_summary["Discount"]*100
discount_summary


An association between discount and margin does not establish causality; product mix, quantity, channel and other variables may differ.


## 7. Top products


In [ ]:
top_products = df.groupby("Product", as_index=False).agg(
    Revenue=("Revenue","sum"), Profit=("Profit","sum")
).nlargest(10, "Revenue")
px.bar(top_products.sort_values("Revenue"), x="Revenue", y="Product",
       orientation="h", title="Top 10 Products by Revenue").show()


## 8. Recommendations
Use observation → implication → action → limitation/follow-up question.
